In [1]:
import jax
import jax.numpy as jnp
import haiku as hk

In [9]:
#Defining the transformer class that inherits from the hk.moduke in the same way that tf.keras.layers does
class BridgeTransformer(hk.Module):
    #Constructor definition, defining all the architecture hyperparams
    def __init__(self, action_dim, d_model, num_heads, num_layers):
        #Calls the parent class hk.Module to make it track the internal params
        super().__init__()
        #Number of different actions in our policy, output of the policy head in our case 35 contract bids
        #plus option to pass, double or redouble so 38 total actions
        self.action_dim = action_dim
        #Dimensions of the internal token representation
        self.d_model = d_model
        #Number of attention heads
        self.num_heads = num_heads
        #number of layers in our encoder transformer stack
        self.num_layers = num_layers


    def _attention_block(self, x, layer_idx):
        # x shape: (batch, 35, d_model)
        
        # Multi-head self attention
        attn_out = hk.MultiHeadAttention(
            num_heads=self.num_heads,
            key_size=self.d_model // self.num_heads,
            model_size=self.d_model,
            #We add weight initialising here using the standard scale of 1 which will follow the layer size
            w_init=hk.initializers.VarianceScaling(1.0),
            #We need to define a different name for each layer or haiku reuses the same params for each layer in the later call
            name=f"attention_{layer_idx}",
        )(x, x, x)
        
        # First residual connection + layer norm
        x = x + attn_out
        x = hk.LayerNorm(
            axis=-1,
            create_scale=True,
            create_offset=True,
            name=f"layer_norm_a_{layer_idx}"
        )(x)
        
        # Feedforward block
        ff = hk.Linear(self.d_model * 4, name=f"ff1_{layer_idx}")(x)
        ff = jax.nn.relu(ff)
        ff = hk.Linear(self.d_model, name=f"ff2_{layer_idx}")(ff)
        
        # Second residual connection + layer norm
        x = x + ff
        x = hk.LayerNorm(
            axis=-1,
            create_scale=True,
            create_offset=True,
            name=f"layer_norm_b_{layer_idx}"
        )(x)
        
        return x
        # output shape: (batch, 35, d_model) — same as input

    
    def __call__(self, x):
    # x arrives with shape (batch_size, 480)
        
        # we split the 480 dims into meaningful parts using the ellipsis operator, making sure we conserve other dimensions
        #we do this split because the hand and context are global features that dont vary across the 35bid slots
        #The history feature does vary across the bid slots and is sequential in nature so we need to seperate
        #before attention in order to help the model attend to meaningful sequence information
        hand    = x[..., :52]       # (batch, 52)  — the 13 cards of the player
        context = x[..., 52:60]     # (batch, 8)   — vulnerability(yes/no) + pre-opening passes
        history = x[..., 60:]       # (batch, 420) — the bidding history
        # we now reshape the bid history into tokens
        # (batch, 420) to a two dimensional vector corresponding to each bid slot and its 12 features (batch, 35, 12)
        #the 12 features being; (was this slot bid on? was it doubled? was it redoubled?) AND by who? (=3*4=12)
        tokens = history.reshape(x.shape[0], 35, 12)
        #We project each token from 12 dims to d_model dims using a fully connected linear layer
        tokens = hk.Linear(self.d_model)(tokens)

        #Now we add learned positional encodings using an embedding layer (this will allow the model to learn meaningful positional information/develop strategies based on position)
        pos_indices = jnp.arange(35)
        pos_embeddings = hk.Embed(35, self.d_model)(pos_indices)
        # pos_embeddings shape: (35, d_model)
        tokens = tokens + pos_embeddings
        # shape still: (batch, 35, d_model)

        # We can now apply transformer layers
        for i in range(self.num_layers):
            tokens = self._attention_block(tokens, layer_idx=i)
        # shape remains unchanged

        # use mean pool to average over all 35 tokens to prepare it for the MLP
        seq_repr = jnp.mean(tokens, axis=1)

        # We concatenate global features with our sequentic history features
        combined = jnp.concatenate([seq_repr, hand, context], axis=-1)

        #Finally we pass the features through a shared MLP to learn non-linear patterns for global and sequential features together
        combined = hk.Linear(self.d_model)(combined)
        combined = jax.nn.relu(combined)

        #Final step= output heads
        #Policy head, Actor maps to 38 logits, 1 for every possible action
        logits = hk.Linear(self.action_dim)(combined)
        #Value head, maps to a single scalar, the Critic's estimated reward
        value  = hk.Linear(1)(combined)

        return logits, jnp.squeeze(value, axis=-1)

In [13]:
forward_pass = hk.without_apply_rng(hk.transform(
    lambda x: BridgeTransformer(
        action_dim=38,
        d_model=256,
        num_heads=8,
        num_layers=3
    )(x)
))

rng = jax.random.PRNGKey(0)
dummy_x = jnp.zeros((4, 480))
params = forward_pass.init(rng, dummy_x)
logits, value = forward_pass.apply(params, dummy_x)

print("logits shape:", logits.shape)   # (4, 38) ✓
print("value shape: ", value.shape)    # (4,)    ✓
param_count = sum(x.size for x in jax.tree_util.tree_leaves(params))
print(f"Total parameters: {param_count:,}")

logits shape: (4, 38)
value shape:  (4,)
Total parameters: 2,472,743


In [ ]:
# import sys
# sys.path.append('/path/to/your/brl')  # only needed if notebook can't find src

from src.models import make_forward_pass

# Test your transformer via the same function the training pipeline uses
forward_pass = make_forward_pass(activation="relu", model_type="Transformer")

rng     = jax.random.PRNGKey(0)
dummy_x = jnp.zeros((4, 480))
params  = forward_pass.init(rng, dummy_x)
logits, value = forward_pass.apply(params, dummy_x)

print("logits shape:", logits.shape)  
print("value shape: ", value.shape)   
print("Integration successful")

# Also verify existing models still work
forward_pass_mlp = make_forward_pass(activation="relu", model_type="DeepMind")
params_mlp = forward_pass_mlp.init(rng, dummy_x)
logits_mlp, value_mlp = forward_pass_mlp.apply(params_mlp, dummy_x)
print("MLP still works:", logits_mlp.shape)  

logits shape: (4, 38)
value shape:  (4,)
Integration successful
MLP still works: (4, 38)
